https://towardsdatascience.com/sentiment-analysis-with-python-part-2-4f71e7bde59a
https://github.com/aaronkub/machine-learning-examples/blob/master/imdb-sentiment-analysis/Sentiment%20Analysis%20Walkthrough%20Part%202.ipynb

In [ ]:
import numpy as np
import pandas as pd
import os
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
TEXTBLOB = False
SAVE_IMAGES = False
Daily = True; Hourly = False
assert(not(Daily & Hourly))
LINEAR_REGRESSION = False

In [ ]:
if (Daily):
    %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"
elif (Hourly):
    %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/InitializationHourly.ipynb"
    df = df.loc[(df["Hour"]==15)]# | (df_part3["Hour"]==15)]


train_start_date = '2020-05-31'
train_end_date = '2021-05-31'
test_start_date = '2021-05-31'
test_end_date = '2021-10-1'
df = df.ffill()

train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date : test_end_date]


press_release_train = []

press_release_train = train["Articles"].to_list()

press_release_test = []
press_release_test = test["Articles"].to_list()
    
target = [1 if i < 233 else 0 for i in range(233)]

In [ ]:
import re

REPLACE_NO_SPACE = re.compile("(\.)|(\;)|(\:)|(\!)|(\?)|(\,)|(\")|(\()|(\))|(\[)|(\])|(\d+)")
REPLACE_WITH_SPACE = re.compile("(<br\s*/><br\s*/>)|(\-)|(\/)")
NO_SPACE = ""
SPACE = " "

def preprocess_press_release(press_release):
    
    press_release = [REPLACE_NO_SPACE.sub(NO_SPACE, line.lower()) for line in press_release]
    press_release = [REPLACE_WITH_SPACE.sub(SPACE, line) for line in press_release]
    
    return press_release

press_release_train_clean = preprocess_press_release(press_release_train)
press_release_test_clean = preprocess_press_release(press_release_test)

# Baseline

In [ ]:
baseline_vectorizer = CountVectorizer(binary=True)
baseline_vectorizer.fit(press_release_train_clean)
X_baseline = baseline_vectorizer.transform(press_release_train_clean)
X_test_baseline = baseline_vectorizer.transform(press_release_test_clean)

X_train, X_val, y_train, y_val = train_test_split(
    X_baseline, target, train_size = 0.75
)

for c in [0.01, 0.05, 0.25, 0.5, 1]:
    
    lr = LogisticRegression(C=c)
    lr.fit(X_train, y_train)
    print ("Accuracy for C=%s: %s" 
           % (c, accuracy_score(y_val, lr.predict(X_val))))
    
#     Accuracy for C=0.01: 0.87472
#     Accuracy for C=0.05: 0.88368
#     Accuracy for C=0.25: 0.88016
#     Accuracy for C=0.5: 0.87808
#     Accuracy for C=1: 0.87648

final_model = LogisticRegression(C=0.05)
final_model.fit(X_baseline, target)
print ("Final Accuracy: %s" 
       % accuracy_score(target, final_model.predict(X_test_baseline)))
# Final Accuracy: 0.88128